# Conjuntos de preguntas de BioASQ por tarea, teniendo todo (lo posible) como nodos del graof 

Una pregunta entra si:

1. Qwen marca al menos una mención.
2. Todas sus menciones enlazan a un nodo.
3. Cada ítem de la respuesta tiene nodo (solo factoid y list).
4. Ningún nodo de la respuesta es una entidad de la pregunta (solo factoid y list).

In [1]:
import json
import re
import sys

import pandas as pd

sys.path.insert(0, "../../../graphrag")
from entidades.linkear_entidades import Linkeador, menciones_de

BIOASQ = "../../preguntas/BioASQ-training14b/training14b.json"
EXTRACCIONES = "extracciones.jsonl"
ARISTAS = "../../grafo_entero/aristas.parquet"
TAREAS = ["factoid", "list", "yesno", "summary"]
CON_RESPUESTA = ["factoid", "list"]
TIPOS_DE_NODO = ["Gene", "Disease", "Drug"]
PASOS = ["diccionario", "segmentos"]
#cuantas preguntas de cada conjunto se sortean con SEED.
PREGUNTAS_POR_TAREA = {"factoid": None, "list": None, "yesno": 100, "summary": 100}
SEED = 42
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 100)

## Carga de las preguntas


In [2]:
with open(BIOASQ) as archivo:
    todas = json.load(archivo)["questions"]

filas = []
for pregunta in todas:
    if pregunta["type"] not in TAREAS:
        continue
    #summary no trae exact_answer: su referencia es ideal_answer
    oro = pregunta.get("exact_answer")
    if pregunta["type"] == "factoid":
        oro = [oro]
    filas.append({"id": pregunta["id"], "tarea": pregunta["type"], "body": pregunta["body"], "oro": oro,
                  "ideal_answer": pregunta["ideal_answer"]})
preguntas = pd.DataFrame(filas)
preguntas["tarea"].value_counts().reindex(TAREAS)

tarea
factoid    1695
list       1130
yesno      1541
summary    1363
Name: count, dtype: int64

## entidades de la pregunta

Las menciones guardadas de Qwen, cada una enlazada con linkear_mencion. Las que no llegan a ningún nodo quedan en sin_enlazar.

In [3]:
extracciones = {}
with open(EXTRACCIONES) as archivo:
    for linea in archivo:
        fila = json.loads(linea)
        extracciones[fila["id_bioasq"]] = fila["respuesta"]
faltan = preguntas[~preguntas["id"].isin(extracciones)]
print(len(extracciones), "extracciones,", len(faltan), "preguntas sin extraer")
assert len(faltan) == 0

linkeador = Linkeador()

entidades, enlaces, sin_enlazar, menciones = [], [], [], []
for pregunta in preguntas.itertuples():
    entidades_de_la_pregunta, enlaces_de_la_pregunta, menciones_sin_nodo, textos = [], [], [], []
    for mencion in menciones_de(extracciones[pregunta.id]):
        textos.append(mencion.texto)
        enlace = linkeador.linkear_mencion(mencion)
        if enlace is None:
            menciones_sin_nodo.append(mencion.texto)
            continue
        label, ids, paso = enlace
        enlaces_de_la_pregunta.append([mencion.texto, label, ids, paso])
        for id_nodo in ids:
            if [label, id_nodo] not in entidades_de_la_pregunta:
                entidades_de_la_pregunta.append([label, id_nodo])
    entidades.append(entidades_de_la_pregunta)
    enlaces.append(enlaces_de_la_pregunta)
    sin_enlazar.append(menciones_sin_nodo)
    menciones.append(textos)
preguntas["menciones"] = menciones
preguntas["entidades"] = entidades
preguntas["enlaces"] = enlaces
preguntas["sin_enlazar"] = sin_enlazar
preguntas["con_mencion"] = preguntas["menciones"].str.len() > 0
preguntas["todas_enlazan"] = preguntas["con_mencion"] & (preguntas["sin_enlazar"].str.len() == 0)

#los enlaces de todas las preguntas por el paso que los hizo
pasos = pd.DataFrame([{"tarea": pregunta.tarea, "paso": enlace[3]} for pregunta in preguntas.itertuples() for enlace in pregunta.enlaces])
pd.crosstab(pasos["paso"], pasos["tarea"], margins=True)

5729 extracciones, 0 preguntas sin extraer


tarea,factoid,list,summary,yesno,All
paso,,,,,
diccionario,706,445,491,1186,2828
segmentos,83,61,65,106,315
All,789,506,556,1292,3143


## oro de factoid y list

Se enlaza solo con el diccionario, porque los segmentos necesitan el tipo para no dar coincidencias falsas.

In [4]:
oro_nodos, oro_con_nodo, n_oro = [], [], []
for pregunta in preguntas.itertuples():
    if pregunta.tarea not in CON_RESPUESTA:
        oro_nodos.append(None)
        oro_con_nodo.append(None)
        n_oro.append(None)
        continue
    nodos_del_oro, con_nodo = [], 0
    for sinonimos in pregunta.oro:
        coincidencias = []
        for sinonimo in sinonimos:
            for tipo in TIPOS_DE_NODO:
                for id_nodo in linkeador.exacto(tipo, [sinonimo]):
                    if [tipo, id_nodo] not in coincidencias:
                        coincidencias.append([tipo, id_nodo])
        nodos_del_oro.append(coincidencias)
        con_nodo += bool(coincidencias)
    oro_nodos.append(nodos_del_oro)
    oro_con_nodo.append(con_nodo)
    n_oro.append(len(pregunta.oro))
preguntas["oro_nodos"] = oro_nodos
preguntas["oro_con_nodo"] = oro_con_nodo
preguntas["n_oro"] = n_oro
preguntas["todo_el_oro_con_nodo"] = preguntas["oro_con_nodo"] == preguntas["n_oro"]
#si un nodo de la respuesta es una entidad de la pregunta, entra al subgrafo por construccion y A1 no mide la seleccion
#(pasa cuando qwen escribe la respuesta como nombre_en_ingles de una mencion generica, cuando la pregunta nombra lo que
#pide, o por un alias ambiguo del diccionario). None en las tareas sin respuesta en nodos
preguntas["respuesta_no_es_entidad"] = [None if nodos is None else not any(nodo in entidades for item in nodos for nodo in item)
                                        for nodos, entidades in zip(preguntas["oro_nodos"], preguntas["entidades"])]

## vecinos de la capa B

Sin vecinos por GDA, INDICATION o DRUG_TARGET el subgrafo de semillas queda vacío.

In [5]:
aristas = pd.read_parquet(ARISTAS)
grupos_bert = set(aristas.loc[aristas["tipo"] == "DISEASE_BERT", "id_a"])
cruzadas = aristas[aristas["tipo"].isin(["GDA", "INDICATION", "DRUG_TARGET"])]
cruzadas = cruzadas[~cruzadas["id_a"].isin(grupos_bert) & ~cruzadas["id_b"].isin(grupos_bert)]
con_vecinos = set(zip(cruzadas["label_a"], cruzadas["id_a"])) | set(zip(cruzadas["label_b"], cruzadas["id_b"]))
preguntas["entidad_con_vecinos_capa_b"] = [any((label, id_nodo) in con_vecinos for label, id_nodo in lista)
                                           for lista in preguntas["entidades"]]

## cuanto se va perdiendo


In [6]:
CONDICIONES = [("1. Qwen marca alguna mención", "con_mencion", TAREAS),
               ("2. todas las menciones enlazan", "todas_enlazan", TAREAS),
               ("3. todos los ítems de la respuesta tienen nodo", "todo_el_oro_con_nodo", CON_RESPUESTA),
               ("4. la respuesta no es una entidad de la pregunta", "respuesta_no_es_entidad", CON_RESPUESTA)]
quedan = pd.Series(True, index=preguntas.index)
filas = {"preguntas en 14b": preguntas.groupby("tarea").size()}
for nombre, columna, tareas in CONDICIONES:
    aplica = preguntas["tarea"].isin(tareas)
    quedan = quedan & (~aplica | preguntas[columna].fillna(False).astype(bool))
    filas[nombre] = quedan.groupby(preguntas["tarea"]).sum().where(pd.Series({t: t in tareas for t in TAREAS}))
preguntas["entra"] = quedan
filas["entran con vecinos de la capa B"] = preguntas[preguntas["entra"]].groupby("tarea")["entidad_con_vecinos_capa_b"].sum()
embudo = pd.DataFrame(filas).loc[TAREAS].T
embudo

tarea,factoid,list,yesno,summary
preguntas en 14b,1695.0,1130.0,1541.0,1363.0
1. Qwen marca alguna mención,1241.0,786.0,1300.0,909.0
2. todas las menciones enlazan,525.0,345.0,493.0,336.0
3. todos los ítems de la respuesta tienen nodo,130.0,55.0,NaN,NaN
4. la respuesta no es una entidad de la pregunta,111.0,51.0,NaN,NaN
entran con vecinos de la capa B,103.0,47.0,475.0,304.0


In [7]:
#de las que entran, el paso mas tardio que necesito alguna de sus menciones
def ult_paso(enlaces):
    return max((enlace[3] for enlace in enlaces), key=PASOS.index)

entran = preguntas[preguntas["entra"]].copy()
entran["necesita"] = entran["enlaces"].map(ult_paso)
pd.crosstab(entran["necesita"], entran["tarea"], margins=True).reindex(PASOS + ["All"])

tarea,factoid,list,summary,yesno,All
necesita,,,,,
diccionario,97,41,285,446,869
segmentos,14,10,51,47,122
All,111,51,336,493,991


In [8]:
#lo que no entra, por la condicion en la que cae: todas las menciones de Qwen por como terminaron, cuantas preguntas por
#tarea caen en cada condicion, las menciones sin nodo mas repetidas y unos ejemplos sorteados con SEED de cada clase,
#para leer a mano por que no se tomo nada y decidir si preocupa
EJEMPLOS = 8

#todas las menciones que marco Qwen en las preguntas de las tres tareas: por diccionario, por segmentos o sin nodo
menciones_qwen = pd.DataFrame([{"tarea": p.tarea, "paso": e[3]} for p in preguntas.itertuples() for e in p.enlaces] +
                              [{"tarea": p.tarea, "paso": "sin nodo"} for p in preguntas.itertuples() for m in p.sin_enlazar])
por_paso = pd.crosstab(menciones_qwen["paso"], menciones_qwen["tarea"], margins=True).reindex(PASOS + ["sin nodo", "All"])
print("menciones de Qwen por como terminaron:")
display(por_paso)
display((por_paso / por_paso.loc["All"]).round(3))

sin_mencion = preguntas[~preguntas["con_mencion"]]
mencion_sin_nodo = preguntas[preguntas["con_mencion"] & ~preguntas["todas_enlazan"]]
oro_sin_nodo = preguntas[preguntas["todas_enlazan"] & preguntas["tarea"].isin(CON_RESPUESTA) & ~preguntas["todo_el_oro_con_nodo"]]

print("\npreguntas que caen en cada condicion, por tarea:")
display(pd.DataFrame({"1. Qwen no marca ninguna mencion": sin_mencion.groupby("tarea").size(),
                      "2. alguna mencion sin nodo": mencion_sin_nodo.groupby("tarea").size(),
                      "3. algun item del oro sin nodo": oro_sin_nodo.groupby("tarea").size()}).reindex(TAREAS).T)

#las menciones sin nodo: cuantas son, cuantas distintas y las mas repetidas
sueltas = pd.Series([m.lower() for lista in mencion_sin_nodo["sin_enlazar"] for m in lista])
print(f"\n{len(sueltas)} menciones sin nodo, {sueltas.nunique()} distintas. las mas repetidas:")
display(sueltas.value_counts().head(25).to_frame("veces").T)

print(f"\n=== 1. Qwen no marca ninguna mencion ({len(sin_mencion)} preguntas)")
for pregunta in sin_mencion.sample(n=EJEMPLOS, random_state=SEED).itertuples():
    print(f"- [{pregunta.tarea}] {pregunta.body}")

print(f"\n=== 2. alguna mencion sin nodo ({len(mencion_sin_nodo)} preguntas)")
for pregunta in mencion_sin_nodo.sample(n=EJEMPLOS, random_state=SEED).itertuples():
    print(f"- [{pregunta.tarea}] {pregunta.body}")
    print(f"    sin nodo: {pregunta.sin_enlazar}    con nodo: {[e[0] for e in pregunta.enlaces]}")

print(f"\n=== 3. algun item del oro sin nodo ({len(oro_sin_nodo)} preguntas)")
for pregunta in oro_sin_nodo.sample(n=EJEMPLOS, random_state=SEED).itertuples():
    sin_nodo = [sinonimos[0] for sinonimos, nodos in zip(pregunta.oro, pregunta.oro_nodos) if not nodos]
    print(f"- [{pregunta.tarea}] {pregunta.body}")
    print(f"    oro sin nodo: {sin_nodo}    entidades de la pregunta: {[e[0] for e in pregunta.enlaces]}")

menciones de Qwen por como terminaron:


tarea,factoid,list,summary,yesno,All
paso,,,,,
diccionario,706,445,491,1186,2828
segmentos,83,61,65,106,315
sin nodo,838,521,659,1014,3032
All,1627,1027,1215,2306,6175


tarea,factoid,list,summary,yesno,All
paso,,,,,
diccionario,0.434,0.433,0.404,0.514,0.458
segmentos,0.051,0.059,0.053,0.046,0.051
sin nodo,0.515,0.507,0.542,0.440,0.491
All,1.000,1.000,1.000,1.000,1.000



preguntas que caen en cada condicion, por tarea:


tarea,factoid,list,yesno,summary
1. Qwen no marca ninguna mencion,454.0,344.0,241.0,454.0
2. alguna mencion sin nodo,716.0,441.0,807.0,573.0
3. algun item del oro sin nodo,395.0,290.0,NaN,NaN



3032 menciones sin nodo, 2492 distintas. las mas repetidas:


,covid-19,thyroid hormone,gene,lb-100,cohesin,tocilizumab,dupilumab,traumatic brain injury,erenumab,fda,...,cardiac regeneration,children,galcanezumab,eptinezumab,genes,drosophila,verubecestat,bevacizumab,cpx-351,gepotidacin
veces,13,12,9,8,7,7,7,6,6,6,...,5,5,5,5,5,4,4,4,4,4



=== 1. Qwen no marca ninguna mencion (1493 preguntas)
- [factoid] What is the aim of the "Radiogenomics Consortium"?
- [list] Please list 2 human diseases caused by a coronavirus.
- [factoid] What is the vector for coxiella burnetii transmission?
- [yesno] Are bacteria in the genus Clostridium facultative anaerobes?
- [summary] What is the Match BAM to VCF method?
- [factoid] How many PML isoforms exist in the human genome?
- [factoid] What fruit causes Jamaican vomiting sickness?
- [summary] Describe Mozart effect.

=== 2. alguna mencion sin nodo (2537 preguntas)
- [yesno] Is SATB1 expressed in thymocytes?
    sin nodo: ['thymocytes']    con nodo: ['SATB1']
- [yesno] Can botulism poisoning of a pregnant woman harm her fetus?
    sin nodo: ['botulism poisoning', 'pregnant woman', 'her fetus']    con nodo: []
- [list] Which proteins are the different members of the NF-kappaB family of transcription factors?
    sin nodo: ['NF-kappaB family of transcription factors']    con nodo: []
- [

In [ ]:

nombre_de = linkeador.nombre_de
pasan_hasta_3 = preguntas["todas_enlazan"] & preguntas["tarea"].isin(CON_RESPUESTA) & preguntas["todo_el_oro_con_nodo"]
excluidas = preguntas[pasan_hasta_3 & ~preguntas["respuesta_no_es_entidad"].fillna(True).astype(bool)]
filas = []
for pregunta in excluidas.itertuples():
    nodos_del_oro = [tuple(nodo) for item in pregunta.oro_nodos for nodo in item]
    coinciden = [(texto, label, i) for texto, label, ids, paso in pregunta.enlaces for i in ids if (label, i) in nodos_del_oro]
    filas.append({"tarea": pregunta.tarea, "id_bioasq": pregunta.id, "pregunta": pregunta.body,
                  "mencion": " | ".join(texto for texto, label, i in coinciden),
                  "nodo": " | ".join(f"{label}:{i} {nombre_de[(label, i)]}" for texto, label, i in coinciden),
                  "respuesta": " || ".join(sinonimos[0] for sinonimos in pregunta.oro),
                  "items_respuesta": len(pregunta.oro)})
excluidas_paso_4 = pd.DataFrame(filas).sort_values(["tarea", "id_bioasq"])
excluidas_paso_4.to_csv("excluidas_paso_4.csv", index=False)
print("=== 4. la respuesta es una entidad de la pregunta:", excluidas_paso_4.groupby("tarea").size().to_dict())
display(excluidas_paso_4[["tarea", "pregunta", "mencion", "nodo", "respuesta"]])

=== 4. la respuesta es una entidad de la pregunta: {'factoid': 19, 'list': 4}


,tarea,pregunta,mencion,nodo,respuesta
8,factoid,What disease is Velcade (bortezomib) mainly used for?,Velcade (bortezomib) is mainly used for,Disease:C0026764 Multiple Myeloma,multiple myeloma
9,factoid,"What memory problems are reported in the "" Gulf war syndrome""",memory problems,Disease:C0751295 Memory Loss,memory loss
2,factoid,What is generic name of drug Adempas?,Adempas,Drug:CID_736700 Riociguat,riociguat
1,factoid,"What is the disease in which patients are sensitive to DNA crosslinking agents, presenting with ...","patients are sensitive to DNA crosslinking agents, presenting with a high frequency of chromosom...",Disease:C0015625 Fanconi Anemia,Fanconi anemia
6,factoid,Which disease is associated with the ectopic expression of the protein encoded by the gene DUX4?,ectopic expression of the protein encoded by the gene DUX4,"Disease:C0238288 Muscular Dystrophy, Facioscapulohumeral",Facioscapulohumeral dystrophy
7,factoid,"Which tumor suppressor is referred to as ""the guardian of the genome""?",the guardian of the genome,Gene:7157 TP53,p53
4,factoid,What is the effect of a defective CLN3 gene?,CLN3 gene,Gene:1201 CLN3,Batten disease
5,factoid,Which gene harbors the mutation T790M?,T790M,Gene:1956 EGFR,EGFR
3,factoid,Which type of myeloma is ixazomib being evaluated for?,myeloma,Disease:C0026764 Multiple Myeloma,Multiple myeloma
10,factoid,Which protein is found to be mutated in Friedreich's ataxia?,protein,Gene:2395 FXN,Frataxin


In [10]:
nombre_de = linkeador.nombre_de

for tarea in TAREAS:
    conjunto = entran[entran["tarea"] == tarea].sort_values("id")
    consultas = {}
    for pregunta in conjunto.itertuples():
        label, id_nodo = pregunta.entidades[0]
        base = re.sub(r"[^a-z0-9]+", "_", nombre_de[(label, id_nodo)].lower()).strip("_")
        nombre = base + "_" + pregunta.id[-6:]
        if nombre in consultas:
            nombre = base + "_" + pregunta.id
        assert nombre not in consultas, f"nombre repetido: {nombre}"
        consultas[nombre] = {"pregunta": pregunta.body, "entidades": pregunta.entidades, "id_bioasq": pregunta.id,
                             "tarea": pregunta.tarea, "enlaces": pregunta.enlaces, "sin_enlazar": pregunta.sin_enlazar,
                             "entidad_con_vecinos_capa_b": bool(pregunta.entidad_con_vecinos_capa_b),
                             "oro": pregunta.oro, "oro_nodos": pregunta.oro_nodos,
                             "ideal_answer": pregunta.ideal_answer}
    with open(f"consultas_{tarea}.json", "w") as archivo:
        json.dump(consultas, archivo, indent=1, ensure_ascii=False)
    print(f"consultas_{tarea}.json", len(consultas))

consultas_factoid.json 111
consultas_list.json 51
consultas_yesno.json 493
consultas_summary.json 336


In [11]:
muestras = {}
for tarea in TAREAS:
    with open(f"consultas_{tarea}.json") as archivo:
        consultas = json.load(archivo)
    nombres = pd.Series(sorted(consultas))
    cuantas = PREGUNTAS_POR_TAREA[tarea]
    elegidos = nombres if cuantas is None else nombres.sample(n=min(cuantas, len(nombres)), random_state=SEED).sort_values()
    muestras[tarea] = {nombre: consultas[nombre] for nombre in elegidos}
    with open(f"muestra_{tarea}.json", "w") as archivo:
        json.dump(muestras[tarea], archivo, indent=1, ensure_ascii=False)
    print(tarea, len(muestras[tarea]), "de", len(consultas))

factoid 111 de 111
list 51 de 51
yesno 100 de 493
summary 100 de 336


In [12]:
def describir_enlace(enlace):
    texto, label, ids, paso = enlace
    return f"{texto} -> {' | '.join(nombre_de[(label, i)] for i in ids)} [{paso}]"

def describir_respuesta(oro, oro_nodos):
    if oro_nodos is None:
        return oro or ""
    return " || ".join(f"{sinonimos[0]} -> {', '.join(nombre_de[(label, i)] for label, i in nodos)}"
                       for sinonimos, nodos in zip(oro, oro_nodos))

for tarea in TAREAS:
    with open(f"consultas_{tarea}.json") as archivo:
        consultas = json.load(archivo)
    filas = []
    for consulta, datos in consultas.items():
        filas.append({"consulta": consulta, "id_bioasq": datos["id_bioasq"], "pregunta": datos["pregunta"],
                      "menciones": " ; ".join(describir_enlace(e) for e in datos["enlaces"]),
                      "entidades": " ; ".join(f"{label}:{i}" for label, i in datos["entidades"]),
                      "pasos": " ; ".join(e[3] for e in datos["enlaces"]),
                      "respuesta": describir_respuesta(datos["oro"], datos["oro_nodos"]),
                      "nodos_respuesta": " ; ".join(f"{label}:{i}" for nodos in (datos["oro_nodos"] or []) for label, i in nodos),
                      "con_vecinos_capa_b": datos["entidad_con_vecinos_capa_b"],
                      "en_muestra": consulta in muestras[tarea]})
    pd.DataFrame(filas).to_csv(f"preguntas_{tarea}.csv", index=False)
    print(f"preguntas_{tarea}.csv", len(filas), "filas,", sum(f["en_muestra"] for f in filas), "en la muestra")

preguntas_factoid.csv 111 filas, 111 en la muestra
preguntas_list.csv 51 filas, 51 en la muestra
preguntas_yesno.csv 493 filas, 100 en la muestra
preguntas_summary.csv 336 filas, 100 en la muestra
